In [1]:
import numpy as np
import pandas as pd
import os
from collections import defaultdict

def calculate_kpi_metrics(df):
    if len(df) == 0:
        return 0, 0, 0
    # Tính 3 chỉ số quan trọng nhất: Số chuyến, Doanh thu, Tốc độ trung vị
    trips = len(df)
    revenue = df["total_amount"].sum()
    speed_median = df["speed_mph"].median() if "speed_mph" in df.columns else 0
    return trips, revenue, speed_median

def clean_month(month, soft_percentiles=True):
    file = f"../../raw/yellow_tripdata_2021-{month:02d}.parquet"
    print(f"--- Đang xử lý tháng {month:02d}: {file} ---")
    
    # Kiểm tra file tồn tại để tránh sập code khi chạy thiếu file
    if not os.path.exists(file):
        print(f"⚠️ Bỏ qua tháng {month:02d}: Không tìm thấy file nguồn.")
        return 0, 0, 0, pd.DataFrame(), 0, {}

    # ĐỌC DỮ LIỆU ĐẦU VÀO
    df = pd.read_parquet(file)

    # 1. Chuẩn hoá datetime
    df["tpep_pickup_datetime"]  = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

    # 2. Chuẩn hoá numeric
    numeric_cols = [
        "passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
        "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
        "congestion_surcharge", "airport_fee", "RatecodeID"
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 3. Tính Trip duration (phút)
    df["trip_duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

    # 4. Tách Flex Fare trips (payment_type == 0)
    df["is_flex_fare"] = (df["payment_type"] == 0)
    metered = df[~df["is_flex_fare"]].copy()
    flex    = df[df["is_flex_fare"]].copy()

    total_rows = len(metered)
    if total_rows == 0:
        return 0, 0, len(flex), pd.DataFrame(), 0, {"month": month, "raw_revenue": 0, "clean_revenue": 0, "revenue_risk": 0, "raw_speed_p50": 0, "clean_speed_p50": 0, "speed_diff": 0}

    # 5. Hard QA (Kiểm tra ID khu vực hợp lệ)
    lookup_file = "../../raw/taxi_zone_lookup.csv"
    if os.path.exists(lookup_file):
        lookup = pd.read_csv(lookup_file)
        valid_ids = lookup["LocationID"].unique()
    else:
        valid_ids = metered["PULocationID"].unique() # Bộ lọc dự phòng nếu thiếu file lookup

    metered["valid_vendor"] = metered["VendorID"].isin([1, 2, 6, 7])
    metered["valid_time"]   = (metered["tpep_dropoff_datetime"] >= metered["tpep_pickup_datetime"]) & \
                              (metered["trip_duration"] > 0) & (metered["tpep_pickup_datetime"].dt.year == 2021)
    metered["valid_passenger"] = metered["passenger_count"].isin([1, 2, 3, 4, 5, 6, 99])
    metered["valid_ratecode"]  = metered["RatecodeID"].isin([1, 2, 3, 4, 5, 6])
    metered["valid_store"]     = metered["store_and_fwd_flag"].isin(["Y", "N"])
    metered["valid_payment"]   = metered["payment_type"].isin([1, 2, 3, 4, 5, 6])
    metered["valid_PULocationID"] = metered["PULocationID"].isin(valid_ids)
    metered["valid_DOLocationID"] = metered["DOLocationID"].isin(valid_ids)

    # 6. Tính tốc độ (mph) và khống chế lỗi chia cho 0
    metered["speed_mph"] = metered["trip_distance"] / ((metered["trip_duration"] / 60))
    metered["valid_speed"] = (metered["speed_mph"] >= 1) & (metered["speed_mph"] <= metered["speed_mph"].quantile(0.999))

    # 7. Soft QA (Xử lý Outlier)
    if soft_percentiles:
        for col in ["trip_distance", "trip_duration", "fare_amount"]:
            lower = max(0, metered[col].quantile(0.0001))
            upper = metered[col].quantile(0.9999)
            metered[f"valid_{col}"] = (metered[col] >= lower) & (metered[col] <= upper)
        
        safe_distance = metered["trip_distance"].clip(lower=0.001)
        price_per_mile = metered["fare_amount"] / safe_distance
        metered["valid_economics"] = ~((metered["trip_distance"] > 5) & (price_per_mile < 0.5))

    # 8. Kiểm định tài chính tài xế
    metered["valid_tip"] = ((metered["payment_type"] == 1) & (metered["tip_amount"] >= 0)) | \
                            ((metered["payment_type"] != 1) & (metered["tip_amount"] == 0))
    metered["valid_extra"] = metered["extra"] >= 0
    metered["valid_mta"]   = metered["mta_tax"].isin([0, 0.5])
    metered["valid_improvement"] = metered["improvement_surcharge"].isin([0, 0.3])
    metered["valid_congestion"]  = metered["congestion_surcharge"].isin([0, 0.75, 2.5])
    metered["valid_tolls"] = metered["tolls_amount"] >= 0
    metered["valid_airport"] = metered["airport_fee"].fillna(0) >= 0

    # 9. Tổng hợp cờ kiểm định
    flag_cols = [c for c in metered.columns if c.startswith("valid_")]
    metered["valid_all"] = metered[flag_cols].all(axis=1)

    # 10. TÍNH BẢNG TỔNG HỢP LỖI QA CỦA THÁNG (Sửa lỗi dán đè mất cột tỷ lệ %)
    qa_summary = []
    for col in flag_cols:
        fail_cnt = (~metered[col]).sum()
        fail_rate = (fail_cnt / total_rows * 100) if total_rows > 0 else 0
        qa_summary.append({
            "rule": col,
            "fail_count": fail_cnt,
            "fail_rate_percent": round(fail_rate, 4)
        })
    qa_summary_df = pd.DataFrame(qa_summary)

    # Phân tách tập dữ liệu Sạch và Lỗi
    clean = metered[metered["valid_all"]].copy()
    bad   = metered[~metered["valid_all"]].copy()

    # 11. PHÂN TÍCH TÁC ĐỘNG QA LÊN KPI GIỮA ĐOẠN GỐC VÀ SẠCH
    raw_trips, raw_rev, raw_speed = calculate_kpi_metrics(metered)
    clean_trips, clean_rev, clean_speed = calculate_kpi_metrics(clean)

    impact_data = {
        "month": month,
        "raw_revenue": raw_rev,
        "clean_revenue": clean_rev,
        "revenue_risk": raw_rev - clean_rev,
        "raw_speed_p50": raw_speed,
        "clean_speed_p50": clean_speed,
        "speed_diff": raw_speed - clean_speed
    }

    # In thông tin kiểm soát tiến độ ra màn hình log (Dọn sạch các lệnh in trùng lặp)
    print(f"📊 Kết quả: Tổng: {total_rows:,} | Sạch: {len(clean):,} | Lỗi: {len(bad):,} | Flex: {len(flex):,}")

    # GHI DỮ LIỆU RA Ổ CỨNG
    os.makedirs("../../processed/clean_data", exist_ok=True)
    os.makedirs("../../processed/bad_data", exist_ok=True)
    os.makedirs("../../processed/flex_fare", exist_ok=True)

    clean_path = f"../../processed/clean_data/clean_2021-{month:02d}.parquet"
    bad_path   = f"../../processed/bad_data/bad_2021-{month:02d}.parquet"
    flex_path  = f"../../processed/flex_fare/flex_2021-{month:02d}.parquet"

    clean.to_parquet(clean_path, index=False)
    bad.to_parquet(bad_path, index=False)
    flex.to_parquet(flex_path, index=False)

    return len(clean), len(bad), len(flex), qa_summary_df, total_rows, impact_data


# =====================================================================
# CHU TRÌNH TỔNG HỢP VÀ ĐÁNH GIÁ CHẤT LƯỢNG TOÀN DIỆN 12 THÁNG (2021)
# =====================================================================
total_clean = 0
total_bad   = 0
total_flex  = 0
total_rows_all = 0
rule_fail_all = defaultdict(int)
impact_results = []

for month in range(1, 13):
    clean_cnt, bad_cnt, flex_cnt, qa_df, total_rows, impact_data = clean_month(month)
    if total_rows == 0: 
        continue

    impact_results.append(impact_data)
    total_clean += clean_cnt
    total_bad   += bad_cnt
    total_flex  += flex_cnt
    total_rows_all += total_rows

    for _, row in qa_df.iterrows():
        rule_fail_all[row["rule"]] += row["fail_count"]

# KIỂM TRA BẢO VỆ: Nếu có dữ liệu mới chạy báo cáo tổng hợp để tránh lỗi KeyError/Empty
if total_rows_all == 0:
    print("\n❌ KHÔNG CÓ DỮ LIỆU ĐỂ TỔNG HỢP. Vui lòng kiểm tra lại đường dẫn thư mục '../../raw/'!")
else:
    # Xuất báo cáo ma trận lỗi QA tích lũy cả năm
    qa_year = pd.DataFrame([
        {
            "rule": rule,
            "fail_count": cnt,
            "fail_rate_percent": round(cnt / total_rows_all * 100, 4)
        }
        for rule, cnt in rule_fail_all.items()
    ]).sort_values("fail_rate_percent", ascending=False)

    print("\n" + "="*60)
    print("📌 BÁO CÁO KIỂM TOÁN CHẤT LƯỢNG TOÀN NĂM (QA SUMMARY - 2021)")
    print("="*60)
    print(qa_year.to_string(index=False))

    print("\n" + "="*60)
    print("📊 THỐNG KÊ SẢN LƯỢNG DÒNG DỮ LIỆU")
    print("="*60)
    print(f" • Tổng dòng clean (metered) 12 tháng: {total_clean:,}")
    print(f" • Tổng dòng bad (metered) 12 tháng:   {total_bad:,}")
    print(f" • Tổng chuyến Flex Fare 12 tháng:     {total_flex:,}")

    # In báo cáo phân tích tác động trực tiếp lên dòng tiền phục vụ viết báo cáo Word
    print("\n" + "="*60)
    print("📈 PHÂN TÍCH TÁC ĐỘNG QA LÊN KPI (QA IMPACT)")
    print("="*60)
    impact_df = pd.DataFrame(impact_results)
    total_risk = impact_df['revenue_risk'].sum()
    print(f" 🔥 Tổng doanh thu rủi ro (Nhiễu toán học) đã loại bỏ cả năm: ${total_risk:,.2f}\n")
    print(impact_df[["month", "revenue_risk", "raw_speed_p50", "clean_speed_p50"]].to_string(index=False))

    # Lưu file csv phân tích tác động làm đầu vào cho Trợ lý ảo / Dự báo doanh thu
    impact_df.to_csv("qa_impact_analysis.csv", index=False)
    print("\n[HỆ THỐNG]: Đã xuất thành công file 'qa_impact_analysis.csv' ra thư mục làm việc hiện tại.")

--- Đang xử lý tháng 01: ../../raw/yellow_tripdata_2021-01.parquet ---
📊 Kết quả: Tổng: 1,271,417 | Sạch: 1,216,266 | Lỗi: 55,151 | Flex: 98,352
--- Đang xử lý tháng 02: ../../raw/yellow_tripdata_2021-02.parquet ---
📊 Kết quả: Tổng: 1,273,463 | Sạch: 1,220,238 | Lỗi: 53,225 | Flex: 98,246
--- Đang xử lý tháng 03: ../../raw/yellow_tripdata_2021-03.parquet ---
📊 Kết quả: Tổng: 1,797,232 | Sạch: 1,721,706 | Lỗi: 75,526 | Flex: 127,920
--- Đang xử lý tháng 04: ../../raw/yellow_tripdata_2021-04.parquet ---
📊 Kết quả: Tổng: 2,043,167 | Sạch: 1,956,905 | Lỗi: 86,262 | Flex: 128,020
--- Đang xử lý tháng 05: ../../raw/yellow_tripdata_2021-05.parquet ---
📊 Kết quả: Tổng: 2,379,813 | Sạch: 2,283,514 | Lỗi: 96,299 | Flex: 127,296
--- Đang xử lý tháng 06: ../../raw/yellow_tripdata_2021-06.parquet ---
📊 Kết quả: Tổng: 2,710,726 | Sạch: 2,594,701 | Lỗi: 116,025 | Flex: 123,538
--- Đang xử lý tháng 07: ../../raw/yellow_tripdata_2021-07.parquet ---
📊 Kết quả: Tổng: 2,691,090 | Sạch: 2,572,559 | Lỗi: 11